# Análise de Tópicos — Discursos dos Presidentes do Brasil

Este notebook usa IA, especificamente BERTopic, para extrair informações gerais sobre quais assuntos são abordados em cada discurso.

## Configuração Inicial:

No projeto, foi utizado **pandas**, **BERTopic** e **POSTGRESQL**.

In [168]:
import pandas as pd
import os

from bertopic import BERTopic
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv() # configurar .env para conectar ao database do postgres

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

query = """
SELECT presidente, n_par, par
FROM speeches;
"""

df = pd.read_sql(query, engine)
df.head()

,presidente,n_par,par
0,Marechal Deodoro da Fonseca,1,"Concidadãos – O povo, o exército e a armada na..."
1,Marechal Deodoro da Fonseca,2,Como resultado imediato desta revolução nacion...
2,Marechal Deodoro da Fonseca,3,"Para comporem esse governo, enquanto a nação s..."
3,Marechal Deodoro da Fonseca,4,"Concidadãos – O governo provisório, simples ag..."
4,Marechal Deodoro da Fonseca,5,No uso das atribuições e faculdades extraordin...


In [169]:
presidentes_republica_velha = [
    "Marechal Deodoro da Fonseca",
    "Floriano Peixoto",
    "Prudente de Morais",
    "Campos Sales",
    "Rodrigues Alves",
    "Affonso Pena",
    "Nilo Peçanha",
    "Hermes da Fonseca",
    "Wenceslau Brás",
    "Epitácio Pessoa",
    "Arthur Bernardes",
    "Washington Luís"
]

Republica_Velha = df[df['presidente'].isin(presidentes_republica_velha)]['par'].tolist()
Todos = df['par'].tolist()

topic_model = BERTopic(embedding_model="paraphrase-multilingual-mpnet-base-v2",min_topic_size=3)

In [170]:
topics, probs = topic_model.fit_transform(Todos)

In [171]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,487,-1_que_de_do_da,"[que, de, do, da, para, em, não, as, se, os]","[Quero convidá-los a visualizar, num futuro nã..."
1,0,30,0_mudar_ousadia_coragem_mas,"[mudar, ousadia, coragem, mas, homem, nossa, s...","[O ser humano não é só realização prática, mas..."
2,1,29,1_me_meus_dever_minha,"[me, meus, dever, minha, soldado, confiança, d...","[No governo do Estado, que foi-me conferido pe..."
3,2,29,2_revolução_movimento_armadas_novembro,"[revolução, movimento, armadas, novembro, repu...","[Surpreendido, quando em missão do meu País no..."
4,3,24,3_ritmo_penso_contudo_enganar,"[ritmo, penso, contudo, enganar, teremos, crei...","[Homem de meu tempo, sei que essa metodologia ..."
...,...,...,...,...,...
107,106,3,106_sobre_adotaram_antecessores_lento,"[sobre, adotaram, antecessores, lento, acertad...","[É política de lento mas seguro resultado, que..."
108,107,3,107_voto_eleitoral_constitucionais_deturpada,"[voto, eleitoral, constitucionais, deturpada, ...",[Ao contrário: a abolição do elemento servil; ...
109,108,3,108_amazônica_benção_cerrado_imensidão,"[amazônica, benção, cerrado, imensidão, semiár...","[É o que espero fazer, com a ajuda de Deus e d..."
110,109,3,109_ambiental_considerações_culpados_gritante,"[ambiental, considerações, culpados, gritante,...",[Defender o equilíbrio ambiental do planeta é ...


In [184]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Definição do Dicionário Amplo (Arsenal de Palavras)
# Adicione ou remova termos conforme a necessidade do seu corpus
dicionario_categorias = {
    "Economia": ["economia", "finanças", "moeda", "impostos", "mercado", "orçamento", "dívida", "tesouro", "inflação", "comércio"],
    "Constituição e Justiça": ["constituição", "leis", "justiça", "judiciário", "tribunal", "direitos", "normas", "legislação", "reforma"],
    "Militar e Defesa": ["exército", "marinha", "aeronáutica", "militar", "forças armadas", "defesa", "guerra", "segurança nacional", "soberania"],
    "Saúde Pública": ["saúde", "hospitais", "medicina", "epidemias", "saneamento", "vacina", "saúde pública", "médicos"],
    "Educação e Cultura": ["educação", "ensino", "escolas", "universidades", "cultura", "ciência", "professor", "alfabetização", "conhecimento"],
    "Política e Estado": ["governo", "estado", "democracia", "república", "partidos", "eleições", "administração pública", "reforma política"],
    "Relações Internacionais": ["diplomacia", "exterior", "tratados", "embaixadas", "nações", "fronteiras", "acordos internacionais"],
    "Trabalho e Social": ["trabalho", "emprego", "social", "previdência", "pobreza", "trabalhadores", "sindicatos", "assistência"],
    "Infraestrutura": ["obras", "transporte", "energia", "comunicações", "estradas", "ferrovias", "portos", "desenvolvimento urbano"]
}

# Pegamos o modelo que já está na memória do seu notebook
model = topic_model.embedding_model.embedding_model

# 2. Gerar Vetores de Conceito (Centróides)
# Para cada categoria, criamos um único vetor que é a média de todas as palavras do arsenal
embeddings_conceitos = {}
nomes_categorias = list(dicionario_categorias.keys())

for categoria, palavras in dicionario_categorias.items():
    # Encode de todas as palavras do arsenal daquela categoria
    word_embeddings = model.encode(palavras)
    # A média (mean) cria um "centro" para o assunto
    embeddings_conceitos[categoria] = np.mean(word_embeddings, axis=0)

# Convertemos para uma matriz para facilitar o cálculo
matriz_conceitos = np.array(list(embeddings_conceitos.values()))

def classificar_por_contexto(docs_list):
    # Se não houver docs representativos, retorna outros
    if not docs_list or len(docs_list) == 0:
        return "Outros"

    # Usamos os 2 primeiros documentos representativos para captar o contexto real
    texto_contexto = " ".join(docs_list[:2])
    doc_embedding = model.encode([texto_contexto])

    # Calcula a similaridade entre o discurso e os conceitos
    similaridades = cosine_similarity(doc_embedding, matriz_conceitos)[0]

    # --- LÓGICA DO THRESHOLD (LIMIAR) ---
    # Se a maior similaridade for menor que 0.35 (valor sugerido),
    # significa que o texto é muito específico ou ruído.
    threshold = 0.35
    idx_melhor = np.argmax(similaridades)

    if similaridades[idx_melhor] < threshold:
        return "Outros"

    return nomes_categorias[idx_melhor]

# 3. Aplicar ao seu DataFrame
df_info = topic_model.get_topic_info()

# Criar a nova coluna analisando os 'Representative_Docs'
df_info['Topico_Geral'] = df_info['Representative_Docs'].apply(classificar_por_contexto)


In [185]:
# Exibir resultados
df_info[['Representative_Docs', 'Topico_Geral']].head(100)

,Representative_Docs,Topico_Geral
0,"[Quero convidá-los a visualizar, num futuro nã...",Política e Estado
1,"[O ser humano não é só realização prática, mas...",Política e Estado
2,"[No governo do Estado, que foi-me conferido pe...",Política e Estado
3,"[Surpreendido, quando em missão do meu País no...",Política e Estado
4,"[Homem de meu tempo, sei que essa metodologia ...",Política e Estado
...,...,...
95,[Ninguém sente mais do que eu a situação penos...,Política e Estado
96,"[É melhor na distribuição de renda, no acesso ...",Política e Estado
97,"[Vinte anos atrás, quando fui eleito president...",Política e Estado
98,[O combate à corrupção e a defesa da ética no ...,Política e Estado
